# Julia in Zasper

Zasper runs any Jupyter kernel. This notebook uses [IJulia](https://github.com/JuliaLang/IJulia.jl),
Julia's kernel. Everything you learned about cells, modes and outputs works the same.

**Setup.** Install Julia from [julialang.org](https://julialang.org), then in Julia run
`using Pkg; Pkg.add("IJulia")`. The kernel then appears in Zasper's Launcher and kernel picker.
This notebook uses only Julia's standard library.

In [ ]:
VERSION

## Basics

In [ ]:
x = 3
y = 2x^2 + 1     # numbers multiply variables directly

In [ ]:
α, β = 0.5, 2.0  # type \alpha then Tab for α
α * β

In [ ]:
squares = [n^2 for n in 1:10]

In [ ]:
sqrt.(squares)   # the dot applies a function to every element

## Multiple dispatch

Julia picks which method to run from the types of *all* the arguments.

In [ ]:
abstract type Shape end
struct Circle <: Shape; r::Float64; end
struct Rect <: Shape; w::Float64; h::Float64; end

area(c::Circle) = π * c.r^2
area(r::Rect) = r.w * r.h

shapes = [Circle(1), Rect(2, 3), Circle(0.5)]
area.(shapes)

## Speed

In [ ]:
function mandelbrot(c; limit = 80)
    z = zero(c)
    for i in 1:limit
        abs2(z) > 4 && return i
        z = z^2 + c
    end
    limit
end

@time grid = [mandelbrot(complex(x, y)) for y in range(-1.2, 1.2, 300), x in range(-2.1, 0.7, 700)];

## Rich display

Define `show` for a MIME type and Julia values display themselves in the notebook. Here, the
Mandelbrot grid becomes an SVG heat map.

In [ ]:
struct Heatmap
    values::Matrix{Int}
    cell::Int
end

function Base.show(io::IO, ::MIME"image/svg+xml", h::Heatmap)
    rows, cols = size(h.values)
    top = maximum(h.values)
    print(io, """<svg xmlns="http://www.w3.org/2000/svg" width="$(cols * h.cell)" height="$(rows * h.cell)" shape-rendering="crispEdges">""")
    for r in 1:rows, c in 1:cols
        t = h.values[r, c] / top
        colour = t == 1 ? "#0b1020" : "hsl($(round(Int, 200 + 160t)),70%,$(round(Int, 20 + 60t))%)"
        print(io, """<rect x="$((c - 1) * h.cell)" y="$((r - 1) * h.cell)" width="$(h.cell)" height="$(h.cell)" fill="$colour"/>""")
    end
    print(io, "</svg>")
end

Heatmap(grid[1:5:end, 1:5:end], 5)

In [ ]:
using Markdown

md"""
### Markdown from Julia

The heat map above has **$(string(length(grid)))** points, computed in a comprehension.
Maths works too: ``z_{n+1} = z_n^2 + c``.
"""

In [ ]:
HTML("<p style='color:#9558b2;font-weight:600'>HTML from Julia</p>")

## Reading the penguins

In [ ]:
using Statistics

header, lines... = readlines("../data/penguins.csv")
columns = Symbol.(split(header, ","))
records = [NamedTuple{Tuple(columns)}(Tuple(split(line, ","))) for line in lines]
complete = filter(r -> all(!=("NA"), values(r)), records)
length(complete)

In [ ]:
for species in sort(unique(r.species for r in complete))
    masses = [parse(Int, r.body_mass_g) for r in complete if r.species == species]
    println(rpad(species, 10), round(mean(masses)), " g  ± ", round(std(masses)))
end

## Errors

Errors show a stack trace, as in Python.

In [ ]:
area("a string")